> Ported from `correlator_notebooks/` (pychfpga API, 8-board corr64 at 3.2 GHz) to corriscope.
> Changes: `pychfpga` -> `corriscope`; `stdout_log_level` -> `stderr_log_level` (the only logging
> keyword `FPGAArray` has); `b.AUTO_BIN_SET` -> `b.CORR.AUTO_BIN_SET` (the original set a plain
> Python attribute on the board object, not the UCORR register; `start_correlators_async` already
> enables automatic bin-set rotation); `skip_partial_integs` -> `skip_first_incomplete` (corriscope's
> name for the same option). Outputs stripped. Edit `hwm` and `n_nyqzone` for your setup.

In [ ]:
import logging
from corriscope.fpga_array import FPGAArray
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9.5, 6)

# Allow all logging level messages to show up
log=logging.getLogger()
log.setLevel(logging.DEBUG)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

    
def _tqdm(iterable, **kwargs):
    return tqdm(iterable, **kwargs) if tqdm is not None else iterable


%matplotlib ipympl 

In [ ]:
import pickle as pkl

In [ ]:
hwm = 'crs 127:1 134:2 129:3 121:4 124:5 133:6 131:7 120:8'# 56:4'
#hwm2 = 'crs 124:5 133:6 131:7 120:8'
#hwm=' '.join((hwm1,hwm2))
#hwm=' '.join((hwm1,hwm2))

ca = await FPGAArray(hwm, mode='corr64', sync_method='trig', sync_source='bp_trig', sync_master=1, sync_master_source='sync_out', sync_master_output=1, stderr_log_level='warn')
#ca = FPGAArray(hwm, mode='corr32', sync_method='irig-b', sync_source='bp_trig', sync_master=1, sync_master_source='irigb_gen', sync_master_output=1, stderr_log_level='warn')

In [ ]:
print(f'Using firmware with timestamp: {await ca.ib[0].get_fpga_firmware_timestamp_async()}')

In [ ]:
# sync fix
for i in ca.ib:
    i.tcpipe.adc_sync()

In [ ]:
#initialize static gain, num captures
Ncap=100
gain=1<<30

channels = [0,1,2,3,4,5,6,7] #channels - used to start data capture on each board. By default use all channels

In [ ]:
n_nyqzone=4

M = 2**14 # samples
fs = 3200 *1e6 # Hz
f_res = fs / M

if n_nyqzone % 2 == 0:
    fnyq = fs * n_nyqzone / 2
else:
    fnyq = fs * (n_nyqzone - 1) / 2

fnyq_sign = ( ( n_nyqzone % 2 ) * 2 - 1 )

freq = fnyq + fnyq_sign * f_res * np.arange(M//2)    


In [ ]:
def select_stream(slot_sel = [], chan_sel = []):
    '''
    empty arrays --> select all
    Want channel 2 on all boards --> slot_sel=[], chan_sel = [2]
    Want channel 2 on boards 3,4 --> slot_sel=[3,4], chan_sel = [2]

    '''
    #Initialize array of all data streams
    slots = np.arange(8)     
    channels = np.arange(8)
    streams = 16 * slots[:, None] + channels 
    

    slot_sel = slot_sel if len(slot_sel) > 0 else slice(None)
    chan_sel = chan_sel if len(chan_sel) > 0 else slice(None)
    
    # select and flatten
    select_streams = np.ravel(streams[slot_sel, chan_sel])
    
    return select_streams

def coord_from_stream(stream):
    slot, chan = stream // 16, stream % 16
    return slot, chan

In [ ]:
# if ever need the mapping of correlator product --> channels being correlated, uncomment the print statement to see the complete list

N = 64
ij_map = np.empty((N, N), dtype=np.intp)
i, j = np.triu_indices(N)
ij_map[i, j] = np.arange(len(i))
ij_map[j, i] = np.arange(len(i))  
    
prod_to_ij = np.stack((i, j), axis=1)  # index with prod_to_ij[p]

for prod in range(len(prod_to_ij)):
    print('Product ', prod, ": ", prod_to_ij[prod])

# Getting FFT streams

In [ ]:
# ADC -> FFT + apply gains -> corr product

In [ ]:
'''streams are the UCAP streams, have a different indexing system than the board channels'''
streams = select_stream()#slot_sel = [0,1])
streams

In [ ]:
len(streams)

In [ ]:
await ca.sync_async()

In [ ]:
r = await ca.get_data_receiver_async()

In [ ]:
await ca.set_channelizers_async(data_source='adc', fft_bypass=False, scaler_bypass=True)
await ca.start_data_capture_async(0.1, source='scaler', mode=0, data_type = 4)
'''MUST MUST MUST SYNC BEFORE READ_RAW FRAMES. OTHERWISE CAPTURES BETWEEN BOARDS WILL NOT BE SIMULTANEOUS'''


In [ ]:
await ca.sync_async()
t, d, c = r.read_raw_frames(ncap=250, split=False, verbose=0)

In [ ]:
'''
note, the output data buffer has the shape (num_channels, num_captures*frames_per_channel, frames_per_channel*length_of_frame)

right now we are capturing in mode=0, ncap = 1: 
expecting 2 frames per channel, 168384 samples each. so the size of the output data should be (num_channels, 1, 2*16834)

'''



In [ ]:
print(d.dtype,d.shape)

In [ ]:

Ts = 1/(fs*1e6) # sample period in seconds
t_frame = Ts*np.arange(M) # frame duration in seconds

In [ ]:
plt.figure(dpi=80)
plt.plot(freq,np.mean(np.abs(d[0,:,:])**2.0,axis=0),alpha=0.5)
plt.plot(freq,np.mean(np.abs(d[8,:,:])**2.0,axis=0),alpha=0.5)
plt.plot(freq,np.mean(np.abs(d[20,:,:])**2.0,axis=0),alpha=0.5)
plt.yscale('log')
plt.show()


plt.figure(dpi=80)
plt.plot(np.abs(np.fft.fftshift(np.fft.ifft(np.mean(np.abs(d[0,:,:])**2,axis=0)))))
plt.show()


plt.figure(dpi=80)
plt.plot(np.abs(np.fft.fftshift(np.fft.ifft(np.mean(d[0,:,:] *d[1,:,:].conj(),axis=0)  ))))
plt.yscale('log')
plt.show()

# Gain comp

In [ ]:
mean_mag = np.mean(np.abs(d[:,:,:]),axis=1)
print(mean_mag)

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), layout = 'tight',dpi=72)
for n in range(8):
    ax[n//4, n%4].set_title(f'slot {n + 1}')
    ax[n//4, n%4].grid()
    for m in range(8):
        ax[n//4, n%4].semilogy(freq, mean_mag[8*n + m], label = f'ADC{m + 1}')
    if n == 0:
        ax[n//4, n%4].legend()
    
#    try:
#    except:
#        continue

fig.suptitle('integrated FFT data for digital gain computation')
fig.supylabel('|FFT| [arb.]')
fig.supxlabel('frequency [MHz]')
# plt.tight_layout()

plt.show()

In [ ]:
target = 1.5*np.sqrt(2)
target_full = target*2**44
gain_theory = target_full/mean_mag

In [ ]:
#Set the gains, and recover the actual gains as stored in the FPGA

gain_actual = np.zeros_like(gain_theory)

for n in range(8):
    for m in range(8):

        print(f'Slot {n + 1}, channelizer {m + 1}')

        # Ensure that complex gains are disabled
        if ca.ib[n].chan[m].SCALER.USE_COMPLEX_GAINS != 0:
            ca.ib[n].chan[m].SCALER.USE_COMPLEX_GAINS = 0

        # Ensure that floating point gains are enabled
        if ca.ib[n].chan[m].SCALER.USE_FLOAT_GAINS != 1:
            ca.ib[n].chan[m].SCALER.USE_FLOAT_GAINS = 1

        # Set the gains; note that SCALER performs additional processing to split gain_theory into
        # linear and logarithmic components (i.e., mantissa and exponent). We therefore need to read
        # the actual gains in the next step, which are directly read from the FPGA.
        ca.ib[n].chan[m].SCALER.set_gain_table(gain_theory[8*n + m], bank = 0, gain_timestamp = None, log_gain = None)

        # Read the actual gains written in the memory bank of the SCALER module in the FPGA.
        # Apply some processing to recover the value in a fully linear unit.
        #gain_actual[8*n + m] = ca.ib[n].chan[m].SCALER.get_gain_table(bank = 0) # --breaks

# gain_actual = gain_actual[:, bin_map]


# 4+4 bit

In [ ]:
await ca.set_channelizers_async(data_source='adc', fft_bypass=False, scaler_bypass=False)
await ca.start_data_capture_async(0.1, source='scaler', mode=0, channels = channels, data_type = 3)
'''MUST MUST MUST SYNC BEFORE READ_RAW FRAMES. OTHERWISE CAPTURES BETWEEN BOARDS WILL NOT BE SIMULTANEOUS'''


In [ ]:
await ca.sync_async()
t, d_scr, c = r.read_raw_frames(ncap=250, format='16+16',split=False, verbose=0)

In [ ]:
d_scr /= 2**12

In [ ]:
d_sclr_mean_mag = np.mean(np.abs(d_scr), axis = 1)

fig, ax = plt.subplots(4, 4, figsize = (20, 12),dpi =100, layout = 'tight')
for n in range(8):
    try:
        ax[n//4, n%4].set_title(f'slot {n + 1}')
        ax[n//4, n%4].grid()
        for m in range(8):
            ax[n//4, n%4].plot(freq, d_sclr_mean_mag[8*n + m], label = f'ADC{m + 1}')
        ax[n//4, n%4].hlines(target, freq[0], freq[-1], colors = 'red', linestyles = 'dashed', label = 'target')
        if n == 0:
            ax[n//4, n%4].legend()
    except:
        continue

fig.suptitle('integrated scaler magnitude')
fig.supylabel('|scaler| [arb.]')
fig.supxlabel('frequency [MHz]')
plt.show()

# Corr product

In [ ]:
await ca.set_channelizers_async(data_source='adc', funcgen_left_shift=0, fft_bypass=False, scaler_bypass=False)#, gains=1<<26)

In [ ]:
await ca.sync_async() # synchronize all boards in the array - important for the corner-turn to work

In [ ]:
for b in ca.ib:
    b.CORR.AUTO_BIN_SET = 1  # rotate the bin set every integration so all bins are covered (start_correlator already does this)
    #b.CORR.BIN_SET = 1 #this is a static configuration of bin sleection

In [ ]:
65536 / 1

In [ ]:
#integ_period=65536 # 16384 is 500 Mbps total from all 4 boards, seems to have packet loss.
integ_period = 16384
await ca.start_correlators_async(integ_period)
cr = await ca.get_corr_receiver_async()


'''new dimention added to bin_map to choose which bins are selected in the round of 4'''
#in_map = np.zeros(8192, bool)
#in_map[cr.bin_map[3].flatten()]=True
#bad_bins = cr.bin_map[:,0,0] # indices of bins that are sacrificed by the correlator for their read cycle
#in_map[bad_bins]=False # Remove corr sacrificial bins

In [ ]:
await ca.sync_async()

In [ ]:
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush=True, soft_integ_period=8, number_of_results=8, skip_first_incomplete=True, verbose=1)
#print(cr.bin_map.shape)        


'''
new documentation of CR object
bin set,
correlator_id, 
bin clocks (array of 0 - 1984. bins repeated at clock times, so each number is repeated 4 times for corr32 and 8 times for corr64),
number of  complex products in each frame (132 for corr32, 260 for corr64)

'''

In [ ]:
#if set SUTO_BIN_SET = 1, get all the bins by summing across results
dd0.shape
dd0_all = np.nansum(dd0,axis = 0)
dd0_all.shape

prod_map = b.CORR.get_ij_to_prod_map(cr.NCHAN)

In [ ]:
in_map=cr.get_good_bin_mask()

In [ ]:
samples=np.arange(8192)
cross = [0,1,7,484,485,493,506,904] #16, 17, 392, 393]


for prod in cross:
    fig, ax = plt.subplots(dpi=72)
    ax.set_title(f'Correlation product {prod} | input {np.where(prod_map==prod)[0][0]} x {np.where(prod_map==prod)[1][0]} ')
    corr_mags = abs(dd0_all[:,prod])
    ax.grid()
    ax.plot(freq[in_map], corr_mags[in_map],'.', ms=5)

    ax.set_xlabel('Freq [MHz]')
    plt.show()


In [ ]:

pkl.dump({"d":dd0_all,"prod_map":prod_map, "gain":gain_theory},open(f'corr_gain_cal_fft_data_nyq{n_nyqzone}.pkl','wb'))

# Get ADC data

In [ ]:
'''streams are the UCAP streams, have a different indexing system than the board channels'''
streams = select_stream()#slot_sel = [0,1])
streams

In [ ]:
len(streams)

In [ ]:
await ca.sync_async()

In [ ]:
r = await ca.get_data_receiver_async()


In [ ]:
await ca.set_channelizers_async(data_source='adc', fft_bypass=True, scaler_bypass=True)
await ca.start_data_capture_async(0.1, source='funcgen', mode=0, data_type = None)


In [ ]:
'''MUST MUST MUST SYNC BEFORE READ_RAW FRAMES. OTHERWISE CAPTURES BETWEEN BOARDS WILL NOT BE SIMULTANEOUS'''
await ca.sync_async()

t, d, c = r.read_raw_frames(ncap=1, format='14', split=False, verbose=0)

In [ ]:
'''
note, the output data buffer has the shape (num_channels, num_captures*frames_per_channel, frames_per_channel*length_of_frame)

right now we are capturing in mode=0, ncap = 1: 
expecting 2 frames per channel, 168384 samples each. so the size of the output data should be (num_channels, 1, 2*16834)

'''

print(d.shape,d.dtype)

In [ ]:
t_adc, d_adc, c_adc = r.read_raw_frames(ncap=1, format='14', split=False, verbose=0)

In [ ]:
frame_idx = 0
n = len(streams)
disconnected_channels = []


for i, a in enumerate(streams):
    if np.max(d[i, frame_idx, :])<300: disconnected_channels.append(i)
if len(disconnected_channels)>0:
    print(f'Channels {disconnected_channels} are disconnected')
else: print(f'All channels are connected')


In [ ]:
frame_idx = 0
n = 3 #len(streams)
disconnected_channels = []

fig, ax = plt.subplots(
    n, 1,
    figsize=(14, 3*n),   # height scales with number of streams
    sharex=True,
    dpi=72
)

# if n==1, ax isn't an array; make it one for uniform code
ax = np.atleast_1d(ax)

for i, a in enumerate(ax):
    a.plot(d[i, frame_idx, :])
    if np.max(d[i, frame_idx, :])<300: disconnected_channels.append(i)
    slot, chan = coord_from_stream(streams[i])
    a.set_title(f"Board {slot}, Channel {chan}")
    a.set_ylabel("Amplitude")
    a.grid(True)

ax[-1].set_xlabel("Sample Index")
#plt.tight_layout()
if len(disconnected_channels)>0:
    print(f'Channels {disconnected_channels} are disconnected')
else: print(f'All channels are connected')
plt.show()


In [ ]:
2**12

plt.close('all')

In [ ]:
frame_idx = 0
n = 1 #len(streams)
disconnected_channels = []

fig, ax = plt.subplots(
    n, 1,
    figsize=(8*1.25, 3*1.25),   # height scales with number of streams
    sharex=True
)

# if n==1, ax isn't an array; make it one for uniform code
ax = np.atleast_1d(ax)

for i, a in enumerate(ax):
    a.plot(d[i, frame_idx, :])
    if np.max(d[i, frame_idx, :])<300: disconnected_channels.append(i)
    slot, chan = coord_from_stream(streams[i])
    a.set_title(f"2-frame ADC capture (10.24 microseconds)")
    a.set_ylabel("Raw Amplitude")
    a.grid(True)

ax[-1].set_xlabel("Sample Index")
#plt.tight_layout()
if len(disconnected_channels)>0:
    print(f'Channels {disconnected_channels} are disconnected')
else: print(f'All channels are connected')
plt.show()


In [ ]:
pkl.dump({"d":d_adc,"streams":streams},open(f'corr_adc_data_nyq{n_nyqzone}.pkl','wb'))

In [ ]:
print()